In [1]:
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "accelerate",
    "soundfile",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
subprocess.run(["apt-get", "install", "-y", "-q", "libsndfile1", "ffmpeg"], check=True)

print("✅ Environment ready")

Reading package lists...
Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
✅ Environment ready


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token_3090")

In [3]:
HF_TOKEN        = secret_value_0

MODEL_ID        = "facebook/mms-1b-all"   # mms-1b is base only — mms-1b-all is ASR-ready
TARGET_LANG     = "ben"                    # Bengali ISO 639-3 code for MMS
DATASET_REPO_ID = "Sanjidh090/Lipi-Ghor-bn-882-SSTT"
OUTPUT_REPO_ID  = "hasans090/mms_inference_lp"

DATASET_AUDIO_FOLDER = "data"
LOCAL_AUDIO_CACHE    = "/kaggle/working/audio_cache"

VERSION           = 1      # ← change before each Save & Run All
FILES_PER_VERSION = 510

OUTPUT_CSV = f"/kaggle/working/mms_lipighor_v{VERSION}.csv"

CHUNK_SEC   = 20    # MMS has no hard window limit, 20s keeps memory stable
OVERLAP_SEC = 2
BATCH_SIZE  = 48    # CTC is much lighter than seq2seq, push it high

import os
os.makedirs(LOCAL_AUDIO_CACHE, exist_ok=True)
print(f"✅ Config ready — v{VERSION}, files {(VERSION-1)*FILES_PER_VERSION+1} to {VERSION*FILES_PER_VERSION}")

✅ Config ready — v1, files 1 to 510


In [4]:
from transformers import Wav2Vec2ForCTC, AutoProcessor
import torch

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, target_lang=TARGET_LANG)

print("Loading model on GPU 0...")
model_0 = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    target_lang            = TARGET_LANG,
    ignore_mismatched_sizes= True,
    torch_dtype            = torch.float16,
)
model_0 = model_0.to("cuda:0")
model_0.load_adapter(TARGET_LANG)   # load Bengali adapter weights
model_0.eval()

print("Loading model on GPU 1...")
model_1 = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID,
    target_lang            = TARGET_LANG,
    ignore_mismatched_sizes= True,
    torch_dtype            = torch.float16,
)
model_1 = model_1.to("cuda:1")
model_1.load_adapter(TARGET_LANG)   # load Bengali adapter weights
model_1.eval()

models = [model_0, model_1]
print(f"✅ Both models loaded with Bengali adapter")
print(f"   GPU 0: {torch.cuda.get_device_name(0)}")
print(f"   GPU 1: {torch.cuda.get_device_name(1)}")

Loading processor...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.ben.safetensors:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Loading model on GPU 1...


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

✅ Both models loaded with Bengali adapter
   GPU 0: Tesla T4
   GPU 1: Tesla T4


In [5]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(
    repo_id   = OUTPUT_REPO_ID,
    repo_type = "dataset",
    private   = True,
    exist_ok  = True,
)
print(f"✅ Repo ready: https://huggingface.co/datasets/{OUTPUT_REPO_ID}")

✅ Repo ready: https://huggingface.co/datasets/hasans090/mms_inference_lp


In [6]:
import os, torch, tempfile, subprocess, soundfile as sf
from pathlib import Path
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, list_repo_files, upload_file

def get_duration(path):
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ], capture_output=True, text=True)
    return float(result.stdout.strip())

def extract_chunk_ffmpeg(audio_path, start_sec, duration_sec, target_sr=16000):
    tmp = tempfile.mktemp(suffix=".wav")
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-t",  str(duration_sec),
        "-i",  str(audio_path),
        "-ar", str(target_sr),
        "-ac", "1", tmp
    ], capture_output=True, check=True)
    return tmp

def dedup_overlap(t1, t2, max_w=8):
    w1, w2 = t1.split(), t2.split()
    for n in range(min(max_w, len(w1), len(w2)), 0, -1):
        if w1[-n:] == w2[:n]:
            return n
    return 0

def list_audio_files_on_hf(dataset_repo, folder, token):
    AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
    all_files  = list_repo_files(dataset_repo, repo_type="dataset", token=token)
    return sorted([
        f for f in all_files
        if f.startswith(folder + "/") and Path(f).suffix.lower() in AUDIO_EXTS
    ])

def download_single_audio(dataset_repo, hf_path, local_dir, token):
    return hf_hub_download(
        repo_id   = dataset_repo,
        filename  = hf_path,
        repo_type = "dataset",
        token     = token,
        local_dir = local_dir,
    )

def push_csv_to_hf(local_csv, output_repo, token):
    upload_file(
        path_or_fileobj = local_csv,
        path_in_repo    = Path(local_csv).name,
        repo_id         = output_repo,
        repo_type       = "dataset",
        token           = token,
    )

def transcribe_on_gpu(model, wav_files, batch_size):
    """CTC-based: logits → argmax → decode. No generate() call."""
    all_texts = []
    for i in range(0, len(wav_files), batch_size):
        batch_files = wav_files[i : i + batch_size]

        arrays = []
        for f in batch_files:
            audio, _ = sf.read(f)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            arrays.append(audio.astype(np.float32))

        inputs = processor(
            arrays,
            sampling_rate  = 16000,
            return_tensors = "pt",
            padding        = True,
        )
        input_values      = inputs.input_values.to(model.device, dtype=torch.float16)
        attention_mask    = inputs.attention_mask.to(model.device)

        with torch.no_grad():
            logits = model(input_values, attention_mask=attention_mask).logits

        # CTC decode: argmax over vocab dimension
        predicted_ids = torch.argmax(logits, dim=-1)
        texts = processor.batch_decode(predicted_ids)
        all_texts.extend(texts)

    return all_texts

def transcribe_dual_gpu(tmp_files, models, batch_size):
    mid   = len(tmp_files) // 2
    half0 = tmp_files[:mid]
    half1 = tmp_files[mid:]

    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(transcribe_on_gpu, models[0], half0, batch_size)
        f1 = ex.submit(transcribe_on_gpu, models[1], half1, batch_size)
        res0 = f0.result()
        res1 = f1.result()

    return res0 + res1

print("✅ Utilities ready")

✅ Utilities ready


In [7]:
hf_audio_paths = list_audio_files_on_hf(DATASET_REPO_ID, DATASET_AUDIO_FOLDER, HF_TOKEN)
TOTAL_ALL = len(hf_audio_paths)

start_idx = (VERSION - 1) * FILES_PER_VERSION
end_idx   = min(VERSION * FILES_PER_VERSION, TOTAL_ALL)
my_files  = hf_audio_paths[start_idx:end_idx]
TOTAL     = len(my_files)

print(f"Total files in repo : {TOTAL_ALL}")
print(f"This version (v{VERSION}): files [{start_idx}:{end_idx}] — {TOTAL} files\n")

def load_existing_csv_from_hf(output_repo, csv_filename, token):
    try:
        path = hf_hub_download(
            repo_id        = output_repo,
            filename       = csv_filename,
            repo_type      = "dataset",
            token          = token,
            force_download = True,
        )
        df = pd.read_csv(path)
        print(f"✅ Loaded from HF — {len(df)} rows done so far")
        return df
    except Exception as e:
        print(f"ℹ️ No existing CSV (starting fresh): {e}")
        return None

hf_df = load_existing_csv_from_hf(OUTPUT_REPO_ID, Path(OUTPUT_CSV).name, HF_TOKEN)

if hf_df is not None:
    done_ids = set(hf_df["id"].astype(str).tolist())
    results  = hf_df.to_dict("records")
    hf_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
elif Path(OUTPUT_CSV).exists():
    done_df  = pd.read_csv(OUTPUT_CSV)
    done_ids = set(done_df["id"].astype(str).tolist())
    results  = done_df.to_dict("records")
    print(f"Resuming from local — {len(done_ids)} done")
else:
    done_ids = set()
    results  = []
    print("Starting fresh")

print(f"\n📊 {len(done_ids)} done / {TOTAL} total — {TOTAL - len(done_ids)} remaining\n")

for i, hf_path in enumerate(my_files):
    file_id = Path(hf_path).stem

    if file_id in done_ids:
        continue

    os.system("clear")
    print(f"{'─'*60}")
    print(f"  [v{VERSION} — {len(results)+1}/{TOTAL}]  {file_id}")
    print(f"{'─'*60}\n")

    try:
        print("⬇  Downloading...")
        audio_path = download_single_audio(DATASET_REPO_ID, hf_path, LOCAL_AUDIO_CACHE, HF_TOKEN)

        total_sec = get_duration(audio_path)
        print(f"⏱  Duration: {total_sec:.0f}s ({total_sec/60:.1f} min)")

        step = CHUNK_SEC - OVERLAP_SEC
        starts, pos = [], 0.0
        while pos < total_sec:
            starts.append(pos)
            if pos + CHUNK_SEC >= total_sec: break
            pos += step
        print(f"🔪  Chunks: {len(starts)}  →  {len(starts)//2} | {len(starts) - len(starts)//2} across 2 GPUs\n")

        tmp_files = []
        for start in starts:
            dur = min(CHUNK_SEC, total_sec - start)
            tmp_files.append(extract_chunk_ffmpeg(audio_path, start, dur))

        chunk_texts = transcribe_dual_gpu(tmp_files, models, BATCH_SIZE)

        for tmp in tmp_files:
            os.remove(tmp)

        for j, text in enumerate(chunk_texts):
            print(f"  chunk {j+1:>3}/{len(starts)}: {text[:70]}")

        words = chunk_texts[0].split() if chunk_texts else []
        for k in range(1, len(chunk_texts)):
            skip = dedup_overlap(chunk_texts[k-1], chunk_texts[k]) if OVERLAP_SEC > 0 else 0
            words.extend(chunk_texts[k].split()[skip:])
        transcript = " ".join(words).strip()

        os.remove(audio_path)
        print(f"\n✅  {len(transcript.split())} words total")

    except Exception as ex:
        transcript = ""
        print(f"\n❌  ERROR: {ex}")

    results.append({"id": file_id, "transcript": transcript})
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    files_done = len(results)
    is_last    = (i == TOTAL - 1)
    if files_done % 10 == 0 or is_last:
        try:
            push_csv_to_hf(OUTPUT_CSV, OUTPUT_REPO_ID, HF_TOKEN)
            print(f"📤  CSV pushed to HF ({files_done} rows) — mms_lipighor_v{VERSION}.csv")
        except Exception as e:
            print(f"⚠️  Push failed: {e}")

print(f"\n✅ Version {VERSION} done — {len(results)} rows")

Total files in repo : 1019
This version (v1): files [0:510] — 510 files



mms_lipighor_v1.csv:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

✅ Loaded from HF — 420 rows done so far

📊 420 done / 510 total — 90 remaining

────────────────────────────────────────────────────────────
  [v1 — 421/510]  OmhGVPBATTY
────────────────────────────────────────────────────────────

⬇  Downloading...


data/OmhGVPBATTY.mp3:   0%|          | 0.00/18.7M [00:00<?, ?B/s]

⏱  Duration: 1313s (21.9 min)
🔪  Chunks: 73  →  36 | 37 across 2 GPUs

  chunk   1/73: লিকু এটিযান বাংলার গ্রামগঞ্জের খবরের সবাইকে স্বাগত জানাচ্ছি তানিয়া ইস
  chunk   2/73: দেশ নিরাপর 9 দক্ষিণ অঞ্চলের জনসভায় তারিক রহমান ভোটের বাকসে জবাব দেবে 
  chunk   3/73: জাতীয় পারতি এখনও দেশের রাজনীতির নিয়ামক শক্তি সরযন্ত্র করে ধ্বংস করা 
  chunk   4/73: ত্ত ভোটের মাছ প্রচার প্রচারণা আর প্রতিশ্রুতি নিয়ে কর্মী সমর্থকদের দৌ 
  chunk   5/73: শুনছিলেন বিয়ারব কেবল সংবাদ ছিোনামোর ি্রিত যাদের কাছে নারীদের সম্মান 
  chunk   6/73: বারতারিখের নির্বাচনের নারি সমাজ অপমানের সমিচিত জবাব দেবে খুলনা ও জসরির
  chunk   7/73: মজন্ত্র হোচ্ছে অভিযোগ করে তারএক রহমান বলেন এরি অংশ হিসাবে ভোঠ গননায় ব
  chunk   8/73: ছোরমা খল মধ্যদূপরে খুল্লার জনস্রোতে তার এক রহমানছেড়ে যাব না ভয় নয ব
  chunk   9/73: মেে মন্চে উঠে হাজারও জনতার ভালোবাসা আর অভিবাদনে শিকতহন ধানের শেষের কাণ
  chunk  10/73: ককমপিত হয়ে উঠে খালিসপুরের সমাবেশ স্থল ও আশপাসের এলেখা খুল্যা সাতখিরা 
  chunk  11/73: তাতেই প্রমাণিত হয় তারা নারীদের আবদ্ধ করতে 

...

data/UzxeZUqQKXc.mp3:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

⏱  Duration: 2780s (46.3 min)
🔪  Chunks: 155  →  77 | 78 across 2 GPUs

  chunk   1/155: মায়অডিয়বুকে শুনছেন হুময় নাহামেদ লিখিত মিসরেলি সিরিস থেকে একটি অপননা
  chunk   2/155: দরজাপুরিয়ে বেরহয়ার বুদ্ধিক কাজ করল না দরজার এক কোনায় আগুন জবল্য ঠিক
  chunk   3/155: বেঁচে থাকার জন্যে শরীরকে মোটামটি ঠিক রাখতে হবে প্রচুর পানী খেতে হবে তা
  chunk   4/155: কেনাযে তাকে আঁটকেছে সে পানীবন্ধ করে দেবে তখন প্রবল তৃষ্ণায় কমডের পানী
  chunk   5/155: দশরীরের জমে থাকা চর্বি থেকে প্রয়োজনীয় খাদ্য সংগ্রহের চেষ্টা করে একজন
  chunk   6/155: তিনি ধরে নিয়েছেন এইভাবে তিনি বেচে থাকতে পারবেন দশ দিন এর বেশী না তবে 
  chunk   7/155: ইকেটিষ্ট হিসাবে অনেকবার সেই জগতে তার ঢোকার ইচ্ছা হয়েছায় ইচ্ছা এখন পূ
  chunk   8/155: তবে তা কম বেছিনের কলের পানী বন্ধ হয়ে গেছে তিনি শেষ পানি কখন খেয়েছেন 
  chunk   9/155: শুরুটা হলো ঘরী দিয়ে মিসরােলী হঠার দেখলেন ঘরির কাটা উল্টদিকে ঘুরছে মিস
  chunk  10/155: হাতে কাগজ কলম নেই ঠিক তিন্টা বাজার সময় ঘরি উল্টদিকে চলা শুরু করেছিল এ
  chunk  11/155: ঘুমভাংলে দেখেন তিনি বাইনকুলার হ

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📤  CSV pushed to HF (510 rows) — mms_lipighor_v1.csv

✅ Version 1 done — 510 rows


In [8]:
import pandas as pd
from huggingface_hub import hf_hub_download, upload_file
from pathlib import Path

TOTAL_VERSIONS = 2
all_dfs = []

for v in range(1, TOTAL_VERSIONS + 1):
    csv_name = f"mms_lipighor_v{v}.csv"
    try:
        path = hf_hub_download(
            repo_id        = OUTPUT_REPO_ID,
            filename       = csv_name,
            repo_type      = "dataset",
            token          = HF_TOKEN,
            force_download = True,
        )
        df = pd.read_csv(path)
        all_dfs.append(df)
        print(f"✅ v{v}: {len(df)} rows")
    except Exception as e:
        print(f"⚠️  v{v} not found — skipping: {e}")

final_df = pd.concat(all_dfs, ignore_index=True)
final_df = final_df.drop_duplicates(subset="id")
final_df = final_df.sort_values("id").reset_index(drop=True)

FINAL_CSV = "/kaggle/working/mms_lipighor_final.csv"
final_df.to_csv(FINAL_CSV, index=False, encoding="utf-8")
print(f"\n📊 Final CSV: {len(final_df)} rows")

upload_file(
    path_or_fileobj = FINAL_CSV,
    path_in_repo    = "mms_lipighor_final.csv",
    repo_id         = OUTPUT_REPO_ID,
    repo_type       = "dataset",
    token           = HF_TOKEN,
)
print("📤 Final CSV pushed — mms_lipighor_final.csv")
print(final_df[["id", "transcript"]].head(10).to_string(index=False))

mms_lipighor_v1.csv:   0%|          | 0.00/44.1M [00:00<?, ?B/s]

✅ v1: 510 rows


mms_lipighor_v2.csv:   0%|          | 0.00/42.8M [00:00<?, ?B/s]

✅ v2: 509 rows

📊 Final CSV: 1019 rows


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📤 Final CSV pushed — mms_lipighor_final.csv
         id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 